# Pretrain Residual CNN in Colab

This notebook is a thin Colab wrapper around the project Python scripts. It does not duplicate the model or training loop inside notebook cells.

Purpose:
- start the CNN warm-start from a clean output directory;
- train the residual CNN with the ISP frozen;
- use the fast L1(Y) + lambda_uv * L1(UV) proxy only as a warm-start;
- evaluate the resulting checkpoint with VIF, NRQM, UNIQUE, L1_Y, and L1_UV;
- save all warm-start artifacts back to Google Drive.

The quality-aligned loss validated by the quality-overfit sanity is used later for E2E training, not for this warm-start pretraining stage.

## Required Google Drive files

Create this folder:

`MyDrive/ISP_colab/cnn_pretrain/`

Upload these files:

- `cnn_pretrain_bundle.zip`
- `train_compact.h5`
- `day_val_mini.bin`
- `day_val_mini.yuv`
- `night_val_mini.bin`
- `night_val_mini.yuv`

If `train_compact.h5` is already present in `MyDrive/ISP_colab/`, the notebook can use it from there too.

Outputs will be written to:

`MyDrive/ISP_colab/cnn_pretrain_outputs/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ISP_colab')
DRIVE_INPUT = DRIVE_ROOT / 'cnn_pretrain'
DRIVE_OUTPUT = DRIVE_ROOT / 'cnn_pretrain_outputs'

LOCAL_REPO = Path('/content/ISP')
LOCAL_TRAIN_H5 = LOCAL_REPO / 'dataset' / 'train_compact.h5'

CLEAR_OLD_OUTPUTS = True

EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-4
LAMBDA_UV = 1.0
EVAL_EVERY = 5
EVAL_MAX_FRAMES = 3
PATIENCE = 4
SEED = 42
USE_AMP = True

NUM_WORKERS = 0

print(f'Drive input folder:  {DRIVE_INPUT}')
print(f'Drive output folder: {DRIVE_OUTPUT}')
print(f'Local workspace:     {LOCAL_REPO}')

## 2. Check GPU

In [ ]:
import subprocess
import torch

try:
    print(subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,memory.total,driver_version',
        '--format=csv'
    ]).decode())
except Exception as exc:
    print(f'nvidia-smi is not available: {exc}')

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('Please switch Colab runtime to GPU before starting CNN warm-start training.')

## 3. Install dependencies

In [ ]:
!pip install -q pyiqa tomli h5py tqdm pillow

## 4. Prepare local workspace

The project code and input files are copied from Drive to `/content` so training does not read compressed HDF5 data directly from Drive.

In [ ]:
import os
import shutil
import time
import zipfile


def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError('None of these paths exist:\n' + '\n'.join(str(p) for p in paths))


code_zip = first_existing(
    DRIVE_INPUT / 'cnn_pretrain_bundle.zip',
    DRIVE_ROOT / 'cnn_pretrain_bundle.zip',
)
train_h5 = first_existing(
    DRIVE_INPUT / 'train_compact.h5',
    DRIVE_ROOT / 'train_compact.h5',
)

mini_files = {}
for name in ['day_val_mini.bin', 'day_val_mini.yuv', 'night_val_mini.bin', 'night_val_mini.yuv']:
    mini_files[name] = first_existing(
        DRIVE_INPUT / name,
        DRIVE_INPUT / 'data' / name,
        DRIVE_ROOT / 'data' / name,
    )

if LOCAL_REPO.exists():
    print(f'Removing old local workspace: {LOCAL_REPO}')
    shutil.rmtree(LOCAL_REPO)
LOCAL_REPO.mkdir(parents=True, exist_ok=True)

if CLEAR_OLD_OUTPUTS and DRIVE_OUTPUT.exists():
    assert DRIVE_OUTPUT.name == 'cnn_pretrain_outputs'
    assert str(DRIVE_OUTPUT).startswith(str(DRIVE_ROOT))
    print(f'Removing old CNN warm-start outputs: {DRIVE_OUTPUT}')
    shutil.rmtree(DRIVE_OUTPUT)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f'Unzipping {code_zip}...', end=' ', flush=True)
t0 = time.time()
with zipfile.ZipFile(code_zip, 'r') as zf:
    zf.extractall(LOCAL_REPO)
print(f'done ({time.time() - t0:.1f}s)')

LOCAL_TRAIN_H5.parent.mkdir(parents=True, exist_ok=True)
print(f'Copying {train_h5.name}...', end=' ', flush=True)
t0 = time.time()
shutil.copy2(train_h5, LOCAL_TRAIN_H5)
print(f'done ({LOCAL_TRAIN_H5.stat().st_size / 1024**3:.2f} GiB, {time.time() - t0:.1f}s)')

(LOCAL_REPO / 'data').mkdir(parents=True, exist_ok=True)
for name, src in mini_files.items():
    dst = LOCAL_REPO / 'data' / name
    print(f'Copying {name}...', end=' ', flush=True)
    t0 = time.time()
    shutil.copy2(src, dst)
    print(f'done ({dst.stat().st_size / 1024**2:.1f} MiB, {time.time() - t0:.1f}s)')

required = [
    LOCAL_REPO / 'scripts' / 'run_pretrain_cnn.py',
    LOCAL_REPO / 'scripts' / 'evaluate_pretrain_checkpoint.py',
    LOCAL_REPO / 'isp' / 'data' / 'dataset_utils.py',
    LOCAL_REPO / 'metrics' / 'vif.py',
    LOCAL_REPO / 'data' / 'imx623.toml',
    LOCAL_REPO / 'dataset' / 'splits_mini.json',
    LOCAL_TRAIN_H5,
] + [LOCAL_REPO / 'data' / name for name in mini_files]

print('\nRequired files:')
for path in required:
    print(f'  {path.relative_to(LOCAL_REPO)}: {"OK" if path.exists() else "MISSING"}')
    if not path.exists():
        raise FileNotFoundError(path)

print('\nWorkspace is ready.')

## 5. Inspect compact training data

In [ ]:
import h5py

with h5py.File(LOCAL_TRAIN_H5, 'r') as f:
    print(f'HDF5: {LOCAL_TRAIN_H5}')
    for key in f.keys():
        print(f'  {key:<8s} shape={f[key].shape} dtype={f[key].dtype}')
    print('attrs:', dict(f.attrs))

## 6. Run fresh pretraining

The script freezes the ISP, trains only the CNN, logs full-frame validation metrics every `EVAL_EVERY` epochs, and saves the best checkpoint to Drive.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-B', 'scripts/run_pretrain_cnn.py',
    '--train-h5', 'dataset/train_compact.h5',
    '--splits-json', 'dataset/splits_mini.json',
    '--config', 'data/imx623.toml',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--lambda-uv', str(LAMBDA_UV),
    '--eval-every', str(EVAL_EVERY),
    '--eval-max-frames', str(EVAL_MAX_FRAMES),
    '--patience', str(PATIENCE),
    '--seed', str(SEED),
    '--num-workers', str(NUM_WORKERS),
    '--device', 'cuda',
    '--output-dir', str(DRIVE_OUTPUT),
    '--cache-ram',
]
if not USE_AMP:
    cmd.append('--no-amp')

print('Running:')
print(' '.join(cmd))
print()

proc = subprocess.Popen(
    cmd,
    cwd=str(LOCAL_REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()

print(f'\n[subprocess exit={ret}]')
if ret != 0:
    raise RuntimeError(f'Pretraining failed with exit code {ret}')

## 7. Final same-split checkpoint evaluation

This evaluates `cnn_pretrained.pth` and an ISP-only baseline on exactly the same mini-val frames.

In [ ]:
eval_cmd = [
    sys.executable, '-B', 'scripts/evaluate_pretrain_checkpoint.py',
    '--ckpt', str(DRIVE_OUTPUT / 'cnn_pretrained.pth'),
    '--splits-json', 'dataset/splits_mini.json',
    '--config', 'data/imx623.toml',
    '--split', 'val',
    '--eval-max-frames', str(EVAL_MAX_FRAMES),
    '--baseline-mode', 'same-split',
    '--device', 'cuda',
    '--output', str(DRIVE_OUTPUT / 'pretrain_eval_metrics.json'),
]

print('Running:')
print(' '.join(eval_cmd))
print()

proc = subprocess.Popen(
    eval_cmd,
    cwd=str(LOCAL_REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()

print(f'\n[subprocess exit={ret}]')
if ret != 0:
    raise RuntimeError(f'Checkpoint evaluation failed with exit code {ret}')

## 8. Verify CNN warm-start outputs

In [ ]:
import json

expected = [
    DRIVE_OUTPUT / 'cnn_pretrained.pth',
    DRIVE_OUTPUT / 'pretrain_history.json',
    DRIVE_OUTPUT / 'pretrain_eval_metrics.json',
]
print('Expected outputs:')
for path in expected:
    print(f'  {path.name:<28s} {"OK" if path.exists() else "MISSING"}')
    if not path.exists():
        raise FileNotFoundError(path)

with open(DRIVE_OUTPUT / 'pretrain_history.json', 'r') as f:
    history = json.load(f)
with open(DRIVE_OUTPUT / 'pretrain_eval_metrics.json', 'r') as f:
    eval_report = json.load(f)

print(f'\nHistory entries: {len(history)}')
print('Last training record:')
print(json.dumps(history[-1], indent=2)[:2500])

print('\nFinal same-split evaluation checks:')
for name, ok in eval_report['checks'].items():
    print(f'  {name:<42s}: {"OK" if ok else "FAIL"}')

print('\nAverage metrics:')
avg = eval_report['average']
base = eval_report['baseline_average']
for metric in ['vif', 'nrqm', 'unique', 'l1_y', 'l1_uv']:
    print(f'  {metric:<7s} {avg[metric]:.6f}  baseline={base[metric]:.6f}  delta={avg[metric] - base[metric]:+.6f}')

print(f'\nAll CNN warm-start outputs are in: {DRIVE_OUTPUT}')